In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, sum as spark_sum, avg, count, rank, row_number, when
from pyspark.sql.window import Window

spark = SparkSession.builder \
    .appName("Pertemuan5-JoinWindowSQL") \
    .master("local[*]") \
    .getOrCreate()
spark.sparkContext.setLogLevel("ERROR")

print("SparkSession siap. Versi Spark:", spark.version)

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/09/17 16:18:01 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


SparkSession siap. Versi Spark: 3.5.9


In [3]:
import numpy as np
import pandas as pd

np.random.seed(7)
kategori_list = ["Elektronik", "Fashion", "Makanan & Minuman", "Kesehatan & Kecantikan", "Rumah Tangga"]

# Tabel referensi/master: target & manager per kategori (data ini relatif statis, jarang berubah)
data_produk = {
    "kategori": kategori_list,
    "target_bulanan": [50000000, 40000000, 30000000, 25000000, 20000000],
    "manager": ["Andi", "Budi", "Citra", "Dewi", "Eka"],
}
df_produk = spark.createDataFrame(pd.DataFrame(data_produk))

# Tabel transaksi: data yang terus bertambah setiap hari
n = 300
data_transaksi = {
    "order_id": [f"O{i}" for i in range(n)],
    "kategori": np.random.choice(kategori_list, size=n),
    "kota": np.random.choice(["Magelang", "Semarang", "Solo"], size=n),
    "pendapatan": np.random.randint(50000, 500000, size=n),
}
df_transaksi = spark.createDataFrame(pd.DataFrame(data_transaksi))

print("df_produk:")
df_produk.show()
print("df_transaksi (5 baris pertama dari total", df_transaksi.count(), "baris):")
df_transaksi.show(5)

df_produk:


+--------------------+--------------+-------+
|            kategori|target_bulanan|manager|
+--------------------+--------------+-------+
|          Elektronik|      50000000|   Andi|
|             Fashion|      40000000|   Budi|
|   Makanan & Minuman|      30000000|  Citra|
|Kesehatan & Kecan...|      25000000|   Dewi|
|        Rumah Tangga|      20000000|    Eka|
+--------------------+--------------+-------+



df_transaksi (5 baris pertama dari total 300 baris):
+--------+--------------------+--------+----------+
|order_id|            kategori|    kota|pendapatan|
+--------+--------------------+--------+----------+
|      O0|        Rumah Tangga|Magelang|    488643|
|      O1|             Fashion|Magelang|    401943|
|      O2|Kesehatan & Kecan...|    Solo|    452308|
|      O3|Kesehatan & Kecan...|Semarang|    421741|
|      O4|        Rumah Tangga|Semarang|    185244|
+--------+--------------------+--------+----------+
only showing top 5 rows



----------------------------------------
Exception occurred during processing of request from ('127.0.0.1', 53070)
Traceback (most recent call last):
  File "/home/azka/.local/share/uv/python/cpython-3.11.16-linux-x86_64-gnu/lib/python3.11/socketserver.py", line 317, in _handle_request_noblock
    self.process_request(request, client_address)
  File "/home/azka/.local/share/uv/python/cpython-3.11.16-linux-x86_64-gnu/lib/python3.11/socketserver.py", line 348, in process_request
    self.finish_request(request, client_address)
  File "/home/azka/.local/share/uv/python/cpython-3.11.16-linux-x86_64-gnu/lib/python3.11/socketserver.py", line 361, in finish_request
    self.RequestHandlerClass(request, client_address, self)
  File "/home/azka/.local/share/uv/python/cpython-3.11.16-linux-x86_64-gnu/lib/python3.11/socketserver.py", line 755, in __init__
    self.handle()
  File "/home/azka/bigdata/lib/python3.11/site-packages/pyspark/accumulators.py", line 295, in handle
    poll(accum_updates)

In [4]:
df_gabung = df_transaksi.join(df_produk, on="kategori", how="left")
df_gabung.show(5)